# Heger- / Heegner-Kohärenz (eabc-Modell)

Validierung für die #Energiedoku (**SageMath 10.5 / 10.8**). Ramanujan-Konstante $e^{\pi\sqrt{163}}$, optional Par-Supermap (Oktovision).

`matrix.singular_values()` in Sage meiden (BLAS / *kernel died*); stattdessen `mpmath.svd` auf der reellen 16×16-Matrix.

In [ ]:
# =============================================================================
# Heger- / Heegner-Kohärenz (eabc-Modell) — #Energiedoku (SageMath 10.5 / 10.8)
# =============================================================================
# Wichtig: *SageMath*-Kernel wählen (rechts in Jupyter/VSCode) — kein reines "Python 3"!
# Sonst: Import klappt nicht, es folgen NameError zu RealField / exp / …
try:
    from sage.version import version as _sage_v
    print("Kernel ok — SageMath", _sage_v)
except ImportError as e:
    raise RuntimeError(
        "Kein Sage: Im Notebook-Menu den Kernel 'SageMath' / 'Sage' wählen (nicht Python 3)."
    ) from e

# --- Zelle 1: 163-Kohärenz (Ramanujan-Konstante) ---
def check_163_coherence():
    RR = RealField(333)
    # pi in RR, sonst bleibt exp symbolisch. Abstand mit .round() für RealElement.
    val = exp(RR(pi) * sqrt(RR(163)))
    diff = abs(val - val.round())
    print("-" * 50)
    print(f"Wert von e^(pi*sqrt(163)):\n{val}")
    print(f"\nEnergetisches Defizit (Lücke): {diff}")
    print("-" * 50)
    return diff

check_163_coherence()

## Optional: Quaternion-Par-Supermap (BV / #Energiedoku)

- **Sage 10.8:** `M.norm()` geht bei Matrizen über `QuaternionAlgebra` nicht (versucht `CDF`-Einbettung).  
- **`to_complex_matrix()`** existiert auf dieser Matrixklasse in der Regel **nicht**.  
- Lösung: Frobenius-Norm per `reduced_norm()` der Einträge; SVD: **`mpmath.svd`** auf einer **Python-float**-16×16-Matrix (Linkregular). **Nicht** `matrix.singular_values()`: dabei kann der **native LAPACK/BLAS**-Pfad in Jupyter/auf macOS den Sage-Kernel beenden (*The kernel died*).

**Kernel:** *SageMath* / Jupyter-Name `sagemath` wählen — **nicht** der reine **Python-3**-Kernel. Wenn VS Code „Kernel wählen“ leer ist: `python3 -m sage.repl.ipython_kernel --install` oder Sage-Installer nutzen.

In [ ]:
# --- Zelle 2: Par-Supermap & Singulärwert-Analyse (eabc-Oktovision) ---
# mpmath.svd: Mittelteil S meist 16x1/1x16; seltener n×n — Ablesen über mpmath_sigma_list ().
import mpmath as mp
mp.dps = 30

H = QuaternionAlgebra(RDF, -1, -1)
t_val = RDF(40.9187)
phase = H([0.5, t_val, t_val**2, sin(t_val)])
Op1 = matrix(H, 2, 2, [[H(17), phase], [phase.conjugate(), H(1)]])
Op2 = matrix(H, 2, 2, [[H(19), phase], [phase.conjugate(), H(1)]])
M = (Op1.conjugate_transpose().tensor_product(Op2.conjugate_transpose())).conjugate_transpose()

def quat_frobenius(mat):
    s = sum(mat[i, j].reduced_norm() for i in range(mat.nrows()) for j in range(mat.ncols()))
    return float(sqrt(s))

def left_regular_4(q):
    tup = q.coefficient_tuple()
    a, b, c, d = float(tup[0]), float(tup[1]), float(tup[2]), float(tup[3])
    return matrix(RDF, 4, 4, [
        [a, -b, -c, -d],
        [b,  a, -d,  c],
        [c,  d,  a, -b],
        [d, -c,  b,  a]
    ])

def to_real_16x16_sage(M4):
    # 4x4-Blöcke reell: block_matrix nutzen (stabil über Sage-10.5/10.8; umgeht set_block-Ecken)
    blocks = [[left_regular_4(M4[i, j]) for j in range(4)] for i in range(4)]
    return block_matrix(RDF, 4, 4, blocks)

def mpmath_sigma_list(S):
    r, c = int(S.rows), int(S.cols)
    if r == 1 and c == 1:
        return [float(S[0, 0])]
    if c == 1:
        return [float(S[i, 0]) for i in range(r)]
    if r == 1:
        return [float(S[0, j]) for j in range(c)]
    if r == c:
        return [float(S[i, i]) for i in range(r)]
    raise ValueError("unerwarteter Singulärfaktor (mpmath.svd)")

def run_svd_analysis(mat_M):
    B = to_real_16x16_sage(mat_M)
    data = [[float(B[i, j]) for j in range(16)] for i in range(16)]
    B_mp = mp.matrix(data)
    _, S_mp, _ = mp.svd(B_mp)
    svals = sorted(mpmath_sigma_list(S_mp), reverse=True)
    s_sum = float(sum(svals)) if svals else 0.0
    sq_sum = float(sum(s**2 for s in svals))
    pur = (sq_sum / (s_sum**2)) if s_sum > 1e-20 else 0.0
    print(f"Frobenius-Norm (manuell): {quat_frobenius(mat_M):.6f}")
    print("--- SVD-Metrik (16×16 eabc-Raum) ---")
    print(f"Top-4-Singulärwerte: {[round(s, 4) for s in svals[:4]]}")
    print(f"Kausale Reinheit (eabc-Metrik): {pur:.8f}")

run_svd_analysis(M)